In [ ]:
import os
!pip install roboflow

from roboflow import Roboflow
rf = Roboflow(api_key=os.environ["ROBOFLOW_API_KEY"])
project = rf.workspace("kj-zy1ya").project("crack-btlnb")
version = project.version(4)
dataset = version.download("yolov8")
                

# CRACK SEGMENTATION 

# Import libraries

In [ ]:
# ============================================================
# Cell 2: Import
# ============================================================
import os, yaml, random, shutil
from pathlib import Path
from collections import Counter, defaultdict

import cv2
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from matplotlib.colors import TABLEAU_COLORS

##  Configuration

In [ ]:
# ============================================================
# Cell 3: Configuration
# ============================================================
DATA_DIR    = Path(r"D:\\Code\\Hu_hong_đường\\crack-4")
SPLITS      = ["train", "valid", "test"]
IMG_EXTS    = {".jpg", ".jpeg", ".png", ".bmp"}
COLORS      = list(TABLEAU_COLORS.values())

# Đọc data.yaml để lấy tên lớp
YAML_PATH = DATA_DIR / "data.yaml"

def load_yaml():
    with open(YAML_PATH, "r", encoding="utf-8") as f:
        return yaml.safe_load(f)

cfg = load_yaml()
CLASS_NAMES = cfg.get("names", [])
NC          = cfg.get("nc", len(CLASS_NAMES))
print(f"[INFO] Số lớp (nc): {NC}")
print(f"[INFO] Tên các lớp : {CLASS_NAMES}")


## Kiểm tra phân bố nhãn (label distribution)

In [ ]:
# ============================================================
# Cell 4: Kiểm tra phân bố nhãn (label distribution)
# ============================================================
def check_label_distribution():
    print("\n" + "="*60)
    print("KIỂM TRA 1: PHÂN BỐ NHÃN THEO TỪNG SPLIT")
    print("="*60)

    all_counts = {}
    for split in SPLITS:
        label_dir = DATA_DIR / split / "labels"
        if not label_dir.exists():
            print(f"[WARN] Không tìm thấy: {label_dir}")
            continue

        counter     = Counter()
        empty_files = []
        invalid_cls = []

        for lbl_file in label_dir.glob("*.txt"):
            lines = lbl_file.read_text(encoding="utf-8").strip().splitlines()
            if not lines:
                empty_files.append(lbl_file.name)
                continue
            for line in lines:
                parts = line.strip().split()
                if not parts:
                    continue
                cls_id = int(parts[0])
                if cls_id < 0 or cls_id >= NC:
                    invalid_cls.append((lbl_file.name, cls_id))
                counter[cls_id] += 1

        all_counts[split] = counter
        total = sum(counter.values())
        print(f"\n[{split.upper()}] Tổng polygon: {total} | File rỗng: {len(empty_files)}")
        for cid in sorted(counter):
            name = CLASS_NAMES[cid] if cid < len(CLASS_NAMES) else f"id_{cid}"
            pct  = counter[cid] / total * 100 if total else 0
            print(f"  Lớp {cid:2d} ({name:20s}): {counter[cid]:6d} ({pct:5.1f}%)")
        if empty_files:
            print(f"  [CẢNH BÁO] File rỗng ({len(empty_files)}): {empty_files[:5]}")
        if invalid_cls:
            print(f"  [LỖI] Class ID ngoài [0,{NC-1}]: {invalid_cls[:5]}")

    # Biểu đồ
    if not all_counts:
        return
    fig, axes = plt.subplots(1, len(all_counts), figsize=(6*len(all_counts), 5))
    if len(all_counts) == 1:
        axes = [axes]
    for ax, (split, counter) in zip(axes, all_counts.items()):
        labels = [CLASS_NAMES[i] if i < len(CLASS_NAMES) else f"id_{i}" for i in sorted(counter)]
        values = [counter[i] for i in sorted(counter)]
        bars = ax.bar(labels, values, color=COLORS[:len(labels)])
        ax.set_title(f"{split} — phân bố nhãn", fontsize=12, fontweight="bold")
        ax.set_ylabel("Số lượng polygon")
        ax.tick_params(axis="x", rotation=30)
        for bar, v in zip(bars, values):
            ax.text(bar.get_x() + bar.get_width()/2,
                    bar.get_height() + max(values)*0.01,
                    str(v), ha="center", va="bottom", fontsize=9)
    plt.tight_layout()
    out = DATA_DIR / "label_distribution.png"
    plt.savefig(str(out), dpi=120)
    plt.show()
    print(f"\n[SAVED] {out}")

check_label_distribution()

## Kiểm tra label có nằm đúng trong ảnh không

In [ ]:
# ============================================================
# Cell 5: Kiểm tra polygon có nằm trong ảnh không
# YOLO-Seg format: class x1 y1 x2 y2 ... xn yn
# Tất cả tọa độ phải trong [0, 1]
# ============================================================
def check_polygon_bounds():
    print("\n" + "="*60)
    print("KIỂM TRA 2: POLYGON CÓ NẰM TRONG ẢNH KHÔNG")
    print("="*60)

    TOL = 1e-4
    issues = defaultdict(list)

    for split in SPLITS:
        label_dir = DATA_DIR / split / "labels"
        if not label_dir.exists():
            continue

        for lbl_file in label_dir.glob("*.txt"):
            for ln, line in enumerate(
                    lbl_file.read_text(encoding="utf-8").strip().splitlines(), 1):
                parts = line.strip().split()
                # YOLO-Seg: 1 class + ít nhất 3 điểm (6 tọa độ) = 7 phần
                if len(parts) < 7:
                    continue
                coords = list(map(float, parts[1:]))
                if len(coords) % 2 != 0:
                    issues[split].append(
                        f"{lbl_file.name} ln{ln}: số tọa độ lẻ ({len(coords)})")
                    continue
                xs = coords[0::2]
                ys = coords[1::2]
                bad_x = [x for x in xs if x < -TOL or x > 1 + TOL]
                bad_y = [y for y in ys if y < -TOL or y > 1 + TOL]
                if bad_x:
                    issues[split].append(
                        f"{lbl_file.name} ln{ln}: x ngoài [0,1] → {bad_x[:3]}")
                if bad_y:
                    issues[split].append(
                        f"{lbl_file.name} ln{ln}: y ngoài [0,1] → {bad_y[:3]}")

        n = len(issues.get(split, []))
        status = "✅ OK" if n == 0 else f"❌ {n} vấn đề"
        print(f"[{split.upper():6s}] {status}")
        for msg in issues.get(split, [])[:10]:
            print(f"  → {msg}")

    return issues

poly_issues = check_polygon_bounds()

## Kiểm tra kích thước ảnh

In [ ]:
# ============================================================
# Cell 6: Kiểm tra kích thước ảnh
# ============================================================
def check_image_sizes():
    print("\n" + "="*60)
    print("KIỂM TRA 3: KÍCH THƯỚC ẢNH")
    print("="*60)

    for split in SPLITS:
        img_dir = DATA_DIR / split / "images"
        if not img_dir.exists():
            continue

        size_counter = Counter()
        corrupt      = []

        for img_path in img_dir.iterdir():
            if img_path.suffix.lower() not in IMG_EXTS:
                continue
            img = cv2.imread(str(img_path))
            if img is None:
                corrupt.append(img_path.name)
                continue
            h, w = img.shape[:2]
            size_counter[(w, h)] += 1

        total = sum(size_counter.values())
        print(f"\n[{split.upper()}] Ảnh hợp lệ: {total} | Ảnh hỏng: {len(corrupt)}")
        for (w, h), cnt in size_counter.most_common(5):
            print(f"  {w}x{h} — {cnt} ảnh ({cnt/total*100:.1f}%)")
        if len(size_counter) > 1:
            print(f"  [CẢNH BÁO] Có {len(size_counter)} kích thước khác nhau!")
        if corrupt:
            print(f"  [LỖI] Ảnh hỏng: {corrupt[:5]}")

check_image_sizes()

In [ ]:
# ============================================================
# Cell 7: Hiển thị polygon segmentation trên ảnh
# ============================================================
def visualize_seg_labels(split="train", n_samples=9, seed=42):
    print("\n" + "="*60)
    print(f"KIỂM TRA 4: HIỂN THỊ POLYGON TRÊN ẢNH [{split}]")
    print("="*60)

    img_dir   = DATA_DIR / split / "images"
    label_dir = DATA_DIR / split / "labels"
    img_files = [p for p in img_dir.iterdir()
                 if p.suffix.lower() in IMG_EXTS]

    random.seed(seed)
    samples = random.sample(img_files, min(n_samples, len(img_files)))

    cols = 3
    rows = (len(samples) + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(cols * 5, rows * 5))
    axes = np.array(axes).flatten()

    for ax, img_path in zip(axes, samples):
        img = cv2.imread(str(img_path))
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        h, w = img.shape[:2]

        lbl_path = label_dir / (img_path.stem + ".txt")
        overlay  = img.copy()

        if lbl_path.exists():
            for line in lbl_path.read_text(encoding="utf-8").strip().splitlines():
                parts = line.strip().split()
                if len(parts) < 7:
                    continue
                cls_id = int(parts[0])
                coords = list(map(float, parts[1:]))
                if len(coords) % 2 != 0:
                    continue
                xs = [x * w for x in coords[0::2]]
                ys = [y * h for y in coords[1::2]]
                pts = np.array(list(zip(xs, ys)), dtype=np.int32)

                color = tuple(int(c * 255) for c in
                              plt.cm.tab10(cls_id % 10)[:3])
                # Vẽ polygon fill bán trong suốt
                cv2.fillPoly(overlay, [pts], color)
                # Vẽ đường viền
                cv2.polylines(img, [pts], isClosed=True,
                              color=color, thickness=2)
                # Label text
                name = CLASS_NAMES[cls_id] if cls_id < len(CLASS_NAMES) \
                       else f"id_{cls_id}"
                cv2.putText(img, name, (pts[0][0], max(pts[0][1]-5, 10)),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 1,
                            cv2.LINE_AA)

        # Blend overlay
        img = cv2.addWeighted(img, 0.7, overlay, 0.3, 0)
        ax.imshow(img)
        ax.set_title(img_path.name[:30], fontsize=8)
        ax.axis("off")

    for ax in axes[len(samples):]:
        ax.axis("off")

    plt.tight_layout()
    out = DATA_DIR / f"seg_viz_{split}.png"
    plt.savefig(str(out), dpi=120)
    plt.show()
    print(f"[SAVED] {out}")

visualize_seg_labels(split="train", n_samples=9)

In [ ]:
# ============================================================
# Cell 8: Kiểm tra ghép cặp ảnh — label
# ============================================================
def check_image_label_pairing():
    print("\n" + "="*60)
    print("KIỂM TRA 5: KHỚP GIỮA ẢNH VÀ FILE NHÃN")
    print("="*60)

    for split in SPLITS:
        img_dir   = DATA_DIR / split / "images"
        label_dir = DATA_DIR / split / "labels"
        if not img_dir.exists():
            continue

        img_stems   = {p.stem for p in img_dir.iterdir()
                       if p.suffix.lower() in IMG_EXTS}
        label_stems = ({p.stem for p in label_dir.glob("*.txt")}
                       if label_dir.exists() else set())

        no_label = img_stems - label_stems
        no_img   = label_stems - img_stems
        paired   = img_stems & label_stems

        print(f"\n[{split.upper():6s}]  Ảnh: {len(img_stems)}"
              f"  |  Label: {len(label_stems)}")
        print(f"   ✅ Khớp đôi    : {len(paired)}")
        if no_label:
            print(f"   ❌ Ảnh thiếu label ({len(no_label)}): "
                  f"{list(no_label)[:5]}")
        if no_img:
            print(f"   ❌ Label thừa ({len(no_img)}): "
                  f"{list(no_img)[:5]}")

check_image_label_pairing()


## XÓA FILE LABEL RỖNG

In [ ]:
# ============================================================
# Cell 9: XÓA FILE LABEL RỖNG
# ============================================================
def delete_empty_labels():
    print("="*60)
    print("XÓA FILE LABEL RỖNG")
    print("="*60)
    
    deleted = {}
    for split in SPLITS:
        label_dir = DATA_DIR / split / "labels"
        if not label_dir.exists():
            continue
            
        empty_count = 0
        for lbl_file in label_dir.glob("*.txt"):
            content = lbl_file.read_text(encoding="utf-8").strip()
            if not content or content == "":
                lbl_file.unlink()
                empty_count += 1
        
        deleted[split] = empty_count
        print(f"[{split.upper():6s}] Xóa {empty_count} file rỗng")
    
    total_deleted = sum(deleted.values())
    print(f"\n✅ Tổng cộng xóa {total_deleted} file label rỗng")
    print("Chạy lại Cell 4 để xác nhận!")

delete_empty_labels()

# Hàm Huấn luyện    

In [ ]:

import time
from datetime import datetime
from ultralytics import YOLO
from pathlib import Path

DATA_DIR = Path(r"D:\\Code\\Hu_hong_đường\\crack-4")
YAML_PATH = DATA_DIR / "data.yaml"

def log_with_time(log_path, msg):
    timestamp = datetime.now().strftime('%Y-%m-%d %H:%M:%S')
    line = f"[{timestamp}] {msg}"
    print(line)
    log_path.parent.mkdir(parents=True, exist_ok=True)
    with open(log_path, "a", encoding="utf-8") as f:
        f.write(line + "\\n")

def train_model(model_name, name_suffix, batch_size=8):
    log_path = DATA_DIR / "runs" / f"seg_train_log_{name_suffix}.txt"
    
    start_dt = datetime.now()
    start_ts = time.time()
    
    def log(msg):
        log_with_time(log_path, msg)
    
    log(f"=== TRAIN {model_name.upper()} ===")
    log(f"Data: {YAML_PATH}")
    log(f"Epochs: 100 | Imgsz: 640 | Batch: {batch_size}")
    log(f"Bắt đầu: {start_dt.strftime('%Y-%m-%d %H:%M:%S')}")
    
    model = YOLO(model_name)
    results = model.train(
        data=str(YAML_PATH),
        epochs=100,
        imgsz=640,
        batch=batch_size,
        patience=20,
        device=0,
        project=str(DATA_DIR / "runs"),
        name=f"crack_seg_{name_suffix}",
        exist_ok=True,
    )
    
    
    elapsed = time.time() - start_ts
    hh, mm, ss = int(elapsed//3600), int((elapsed%3600)//60), int(elapsed%60)
    
    log(f"=== XONG {model_name.upper()} ===")
    log(f"Thời gian: {hh:02d}h {mm:02d}m {ss:02d}s")
    if hasattr(results, "results_dict"):
        for k, v in results.results_dict.items():
            if isinstance(v, float):
                log(f"{k:<30s}: {v:.4f}")
    log(f"Best: runs/crack_seg_{name_suffix}/weights/best.pt")
    
    return results

# Train YOLOv8n-seg

In [ ]:

# Cell 10: Train YOLOv8n-seg (nhỏ nhất, nhanh nhất)
print("=== CELL 10: YOLOv8n-seg ===")
train_model("yolov8n-seg.pt", "nano")

In [ ]:
# ============================================================
# Cell 10: RESUME HUẤN LUYỆN YOLOv8n-seg
#          Tiếp tục từ checkpoint cuối cùng (last.pt)
# ============================================================
import time
from datetime import datetime
from pathlib import Path
from ultralytics import YOLO

# ── Đường dẫn ──────────────────────────────────────────────
DATA_DIR   = Path(r"D:\Code\Hu_hong_đường\crack-4")
YAML_PATH  = DATA_DIR / "data.yaml"
SAVE_DIR   = DATA_DIR / "runs" / "crack_seg_nano"
LAST_PT    = SAVE_DIR / "weights" / "last.pt"
LOG_PATH   = SAVE_DIR / "train_log.txt"
LOG_PATH.parent.mkdir(parents=True, exist_ok=True)

# ── Helper log ─────────────────────────────────────────────
def log(msg):
    line = f"[{datetime.now().strftime('%Y-%m-%d %H:%M:%S')}] {msg}"
    print(line)
    with open(LOG_PATH, "a", encoding="utf-8") as f:
        f.write(line + "\n")

# ── Kiểm tra checkpoint tồn tại ────────────────────────────
if LAST_PT.exists():
    log(f"✅ Tìm thấy checkpoint: {LAST_PT}")
    resume_path = str(LAST_PT)
else:
    log("⚠️  Không tìm thấy last.pt → train từ đầu với yolov8n-seg.pt")
    resume_path = None

# ── Bắt đầu ────────────────────────────────────────────────
start_dt = datetime.now()
start_ts = time.time()

log("=" * 55)
log("RESUME / START TRAIN YOLOv8n-seg")
log(f"  Checkpoint : {resume_path or 'yolov8n-seg.pt (mới)'}")
log(f"  Data       : {YAML_PATH}")
log(f"  Save dir   : {SAVE_DIR}")
log(f"  Bắt đầu   : {start_dt.strftime('%Y-%m-%d %H:%M:%S')}")
log("=" * 55)

# ── Load model ─────────────────────────────────────────────
if resume_path:
    # Resume: YOLO tự đọc lại config + epoch đã chạy từ last.pt
    model   = YOLO(resume_path)
    results = model.train(resume=True)
else:
    # Chưa có checkpoint → train mới
    model = YOLO("yolov8n-seg.pt")
    results = model.train(
        data      = str(YAML_PATH),
        epochs    = 100,
        imgsz     = 640,
        batch     = 8,
        patience  = 20,
        device    = 0,
        project   = str(DATA_DIR / "runs"),
        name      = "crack_seg_nano",
        exist_ok  = True,
        # Augmentation
    )

# ── Log kết thúc ───────────────────────────────────────────
elapsed = time.time() - start_ts
hh = int(elapsed // 3600)
mm = int((elapsed % 3600) // 60)
ss = int(elapsed % 60)

log("=" * 55)
log("KẾT THÚC HUẤN LUYỆN")
log(f"  Kết thúc  : {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
log(f"  Thời gian : {hh:02d}h {mm:02d}m {ss:02d}s")

if hasattr(results, "results_dict"):
    for k, v in results.results_dict.items():
        if isinstance(v, float):
            log(f"  {k:<35s}: {v:.4f}")

log(f"  Best model : {SAVE_DIR / 'weights' / 'best.pt'}")
log(f"  Log file   : {LOG_PATH}")
log("=" * 55)

# Train YOLOv8m-seg (cân bằng)

In [ ]:
# Cell 11: Train YOLOv8m-seg (cân bằng)
print("\\n=== CELL 11: YOLOv8m-seg ===")
train_model("yolov8m-seg.pt", "medium", batch_size=4)

In [ ]:
# ============================================================
# Cell 10: RESUME HUẤN LUYỆN YOLOv8n-seg
#          Tiếp tục từ checkpoint cuối cùng (last.pt)
# ============================================================
import time
from datetime import datetime
from pathlib import Path
from ultralytics import YOLO

# ── Đường dẫn ──────────────────────────────────────────────
DATA_DIR   = Path(r"D:\Code\Hu_hong_đường\crack-4")
YAML_PATH  = DATA_DIR / "data.yaml"
SAVE_DIR   = DATA_DIR / "runs" / "crack_seg_medium"
LAST_PT    = SAVE_DIR / "weights" / "last.pt"
LOG_PATH   = SAVE_DIR / "train_log.txt"
LOG_PATH.parent.mkdir(parents=True, exist_ok=True)

# ── Helper log ─────────────────────────────────────────────
def log(msg):
    line = f"[{datetime.now().strftime('%Y-%m-%d %H:%M:%S')}] {msg}"
    print(line)
    with open(LOG_PATH, "a", encoding="utf-8") as f:
        f.write(line + "\n")

# ── Kiểm tra checkpoint tồn tại ────────────────────────────
if LAST_PT.exists():
    log(f"✅ Tìm thấy checkpoint: {LAST_PT}")
    resume_path = str(LAST_PT)
else:
    log("⚠️  Không tìm thấy last.pt → train từ đầu với yolov8n-seg.pt")
    resume_path = None

# ── Bắt đầu ────────────────────────────────────────────────
start_dt = datetime.now()
start_ts = time.time()

log("=" * 55)
log("RESUME / START TRAIN YOLOv8n-seg")
log(f"  Checkpoint : {resume_path or 'yolov8n-seg.pt (mới)'}")
log(f"  Data       : {YAML_PATH}")
log(f"  Save dir   : {SAVE_DIR}")
log(f"  Bắt đầu   : {start_dt.strftime('%Y-%m-%d %H:%M:%S')}")
log("=" * 55)

# ── Load model ─────────────────────────────────────────────
if resume_path:
    # Resume: YOLO tự đọc lại config + epoch đã chạy từ last.pt
    model   = YOLO(resume_path)
    results = model.train(resume=True)
else:
    # Chưa có checkpoint → train mới
    model = YOLO("yolov8n-seg.pt")
    results = model.train(
        data      = str(YAML_PATH),
        epochs    = 100,
        imgsz     = 640,
        batch     = 8,
        patience  = 20,
        device    = 0,
        project   = str(DATA_DIR / "runs"),
        name      = "crack_seg_nano",
        exist_ok  = True,
        # Augmentation
    )

# ── Log kết thúc ───────────────────────────────────────────
elapsed = time.time() - start_ts
hh = int(elapsed // 3600)
mm = int((elapsed % 3600) // 60)
ss = int(elapsed % 60)

log("=" * 55)
log("KẾT THÚC HUẤN LUYỆN")
log(f"  Kết thúc  : {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
log(f"  Thời gian : {hh:02d}h {mm:02d}m {ss:02d}s")

if hasattr(results, "results_dict"):
    for k, v in results.results_dict.items():
        if isinstance(v, float):
            log(f"  {k:<35s}: {v:.4f}")

log(f"  Best model : {SAVE_DIR / 'weights' / 'best.pt'}")
log(f"  Log file   : {LOG_PATH}")
log("=" * 55)

# Train YOLOv8s-seg (nhanh vừa)

In [ ]:
# Cell 12: Train YOLOv8s-seg (nhanh vừa)
print("\\n=== CELL 12: YOLOv8s-seg ===")
train_model("yolov8s-seg.pt", "small", batch_size=8)

In [ ]:
# ============================================================
# Cell 10: RESUME HUẤN LUYỆN YOLOv8n-seg
#          Tiếp tục từ checkpoint cuối cùng (last.pt)
# ============================================================
import time
from datetime import datetime
from pathlib import Path
from ultralytics import YOLO

# ── Đường dẫn ──────────────────────────────────────────────
DATA_DIR   = Path(r"D:\Code\Hu_hong_đường\crack-4")
YAML_PATH  = DATA_DIR / "data.yaml"
SAVE_DIR   = DATA_DIR / "runs" / "crack_seg_small"
LAST_PT    = SAVE_DIR / "weights" / "last.pt"
LOG_PATH   = SAVE_DIR / "train_log.txt"
LOG_PATH.parent.mkdir(parents=True, exist_ok=True)

# ── Helper log ─────────────────────────────────────────────
def log(msg):
    line = f"[{datetime.now().strftime('%Y-%m-%d %H:%M:%S')}] {msg}"
    print(line)
    with open(LOG_PATH, "a", encoding="utf-8") as f:
        f.write(line + "\n")

# ── Kiểm tra checkpoint tồn tại ────────────────────────────
if LAST_PT.exists():
    log(f"✅ Tìm thấy checkpoint: {LAST_PT}")
    resume_path = str(LAST_PT)
else:
    log("⚠️  Không tìm thấy last.pt → train từ đầu với yolov8n-seg.pt")
    resume_path = None

# ── Bắt đầu ────────────────────────────────────────────────
start_dt = datetime.now()
start_ts = time.time()

log("=" * 55)
log("RESUME / START TRAIN YOLOv8n-seg")
log(f"  Checkpoint : {resume_path or 'yolov8n-seg.pt (mới)'}")
log(f"  Data       : {YAML_PATH}")
log(f"  Save dir   : {SAVE_DIR}")
log(f"  Bắt đầu   : {start_dt.strftime('%Y-%m-%d %H:%M:%S')}")
log("=" * 55)

# ── Load model ─────────────────────────────────────────────
if resume_path:
    # Resume: YOLO tự đọc lại config + epoch đã chạy từ last.pt
    model   = YOLO(resume_path)
    results = model.train(resume=True)
else:
    # Chưa có checkpoint → train mới
    model = YOLO("yolov8n-seg.pt")
    results = model.train(
        data      = str(YAML_PATH),
        epochs    = 100,
        imgsz     = 640,
        batch     = 8,
        patience  = 20,
        device    = 0,
        project   = str(DATA_DIR / "runs"),
        name      = "crack_seg_nano",
        exist_ok  = True,
        # Augmentation
    )

# ── Log kết thúc ───────────────────────────────────────────
elapsed = time.time() - start_ts
hh = int(elapsed // 3600)
mm = int((elapsed % 3600) // 60)
ss = int(elapsed % 60)

log("=" * 55)
log("KẾT THÚC HUẤN LUYỆN")
log(f"  Kết thúc  : {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
log(f"  Thời gian : {hh:02d}h {mm:02d}m {ss:02d}s")

if hasattr(results, "results_dict"):
    for k, v in results.results_dict.items():
        if isinstance(v, float):
            log(f"  {k:<35s}: {v:.4f}")

log(f"  Best model : {SAVE_DIR / 'weights' / 'best.pt'}")
log(f"  Log file   : {LOG_PATH}")
log("=" * 55)

In [ ]:

# Cell 10: Train YOLOv8n-seg (nhỏ nhất, nhanh nhất)
print("=== CELL 10: YOLOv8n-seg ===")
# train_model("yolov8n-seg.pt", "nano")

# Cell 11: Train YOLOv8m-seg (cân bằng)
print("\\n=== CELL 11: YOLOv8m-seg ===")
# train_model("yolov8m-seg.pt", "medium", batch_size=4)

# Cell 12: Train YOLOv8s-seg (nhanh vừa)
print("\\n=== CELL 12: YOLOv8s-seg ===")
# train_model("yolov8s-seg.pt", "small", batch_size=8)

print("\\n✅ Thêm comment để chạy từng model. Bắt đầu từ nano!")

#

# yolov8s-seg.pt

In [ ]:
from ultralytics import YOLO

model = YOLO("yolov8s-seg.pt")
model.train(
    data=r"D:\Code\Hu_hong_đường\crack-4\data.yaml",
    epochs=180,
    patience=40,
    batch=4,
    imgsz=640,
    device=0,
    workers=8,
    project=r"D:\Code\Hu_hong_đường\crack-4\runs",
    name="crack_seg_small_allin",
    exist_ok=True,

    pretrained=True,
    optimizer="AdamW",
    lr0=0.003,
    lrf=0.01,
    weight_decay=0.0007,
    warmup_epochs=5.0,
    warmup_momentum=0.8,
    warmup_bias_lr=0.1,
    cos_lr=True,
    close_mosaic=20,

    amp=True,
    deterministic=True,
    seed=0,

    overlap_mask=True,
    mask_ratio=2,

    box=7.5,
    cls=0.3,
    dfl=1.5,

    hsv_h=0.01,
    hsv_s=0.4,
    hsv_v=0.25,
    degrees=2.0,
    translate=0.05,
    scale=0.2,
    shear=0.0,
    perspective=0.0,
    flipud=0.0,
    fliplr=0.5,

    mosaic=0.3,
    mixup=0.0,
    cutmix=0.0,
    copy_paste=0.0,
    erasing=0.0,
    plots=True
)

In [ ]:
from ultralytics import YOLO

model = YOLO("yolov8s-seg.pt")

results = model.train(
    task="segment",
    data=r"D:\Code\Hu_hong_đường\crack-4\data.yaml",
    epochs=200,
    patience=20,
    batch=16,
    imgsz=640,
    device="0",
    workers=4,
    project=r"D:\Code\Hu_hong_đường\crack-4\runs",
    name="crack_seg_small_1",
    exist_ok=True,
    pretrained=True,
    optimizer="auto",
    verbose=True,
    seed=0,
    deterministic=True,
    single_cls=False,
    rect=False,
    cos_lr=False,
    close_mosaic=10,
    amp=True,
    fraction=1.0,
    profile=False,
    freeze=None,
    multi_scale=False,
    overlap_mask=True,
    mask_ratio=4,
    dropout=0.0,
    val=True,
    split="val",
    save=True,
    save_period=-1,
    save_json=False,
    iou=0.7,
    max_det=300,
    half=False,
    plots=True,
    lr0=0.01,
    lrf=0.01,
    momentum=0.937,
    weight_decay=0.0005,
    warmup_epochs=3.0,
    warmup_momentum=0.8,
    warmup_bias_lr=0.0,
    box=7.5,
    cls=0.5,
    dfl=1.5,
    nbs=64,
    hsv_h=0.015,
    hsv_s=0.7,
    hsv_v=0.4,
    degrees=0.0,
    translate=0.1,
    scale=0.5,
    shear=0.0,
    perspective=0.0,
    flipud=0.0,
    fliplr=0.5,
    bgr=0.0,
    mosaic=1.0,
    mixup=0.0,
    cutmix=0.0,
    copy_paste=0.0,
    copy_paste_mode="flip",
    auto_augment="randaugment",
    erasing=0.4,
)

#  Merge hard negatives vào dataset gốc crack-4 + Train lại

In [ ]:
import shutil
from pathlib import Path

# ===== ĐƯỜNG DẪN =====
ROOT     = Path(r"D:\Code\Hu_hong_đường\crack-4")
NEG_IMGS = Path(r"D:\Code\Hu_hong_đường\crack-4\hard_negative_from_prenn\images\train")
NEG_LBLS = Path(r"D:\Code\Hu_hong_đường\crack-4\hard_negative_from_prenn\labels\train")

# Đích đúng với data.yaml gốc (train/images, train/labels)
DEST_IMG = ROOT / "train" / "images"
DEST_LBL = ROOT / "train" / "labels"
DEST_IMG.mkdir(parents=True, exist_ok=True)
DEST_LBL.mkdir(parents=True, exist_ok=True)

# ===== COPY =====
copied = 0
skipped = 0
for img_path in sorted(NEG_IMGS.glob("*.jpg")):
    lbl_path = NEG_LBLS / img_path.with_suffix(".txt").name
    dest_img = DEST_IMG / img_path.name
    dest_lbl = DEST_LBL / img_path.with_suffix(".txt").name

    if dest_img.exists():
        skipped += 1
        continue

    shutil.copy2(img_path, dest_img)
    shutil.copy2(lbl_path, dest_lbl)
    copied += 1

print(f"✅ Copy xong! Thêm: {copied} | Bỏ qua trùng: {skipped}")

# ===== KIỂM TRA SAU MERGE =====
train_imgs = list(DEST_IMG.glob("*.jpg"))
train_lbls = list(DEST_LBL.glob("*.txt"))
nonempty   = [f for f in train_lbls if f.read_bytes()]
empty      = [f for f in train_lbls if not f.read_bytes()]

print(f"\n{'='*55}")
print(f"📊 TRAIN SPLIT SAU MERGE:")
print(f"  Tổng ảnh:           {len(train_imgs)}")
print(f"  Crack (label thật): {len(nonempty)}")
print(f"  Negative (rỗng):    {len(empty)}")
if len(train_imgs) > 0:
    r = len(nonempty) / len(train_imgs) * 100
    print(f"  Tỉ lệ crack:neg   = {r:.0f}%:{100-r:.0f}%")
print(f"{'='*55}")

# Cảnh báo nếu quá ít crack
if len(nonempty) < len(empty):
    print("⚠️  WARNING: Negative nhiều hơn crack!")
    print("   Thêm ảnh crack thật hoặc giảm bớt negative!")
else:
    print("✅ Tỉ lệ hợp lý — sẵn sàng train!")

In [ ]:
from ultralytics import YOLO
import cv2
import os
from pathlib import Path

MODEL_PATH   = r"D:\Code\Hu_hong_đường\crack-4\runs\crack_seg_small_1\weights\best.pt"
VIDEO_PATH   = r"D:\Code\Hu_hong_đường\crack-4\runs\Vết nứt trên đường đèo Prenn Đà Lạt do đâu- - YouTube.mp4"
OUTPUT_ROOT  = r"D:\Code\Hu_hong_đường\crack-4\hard_negative_from_prenn"

CONF         = 0.25   # ⬇ thấp hơn để bắt nhiều FP hơn
EVERY_N      = 3      # ⬇ 8→3: lấy dày hơn gấp ~2.7x
SAVE_CLEAN   = True   # ✅ Lưu cả frame KHÔNG có prediction (clean background)
CLEAN_RATIO  = 3      # Cứ 3 frame clean thì lưu 1 (tránh trùng quá nhiều)
TARGET       = 500    # Dừng khi đủ 500

print("🚀 Hard Negative Miner v2 — target 500+")

img_dir = Path(OUTPUT_ROOT) / "images" / "train"
lbl_dir = Path(OUTPUT_ROOT) / "labels" / "train"
img_dir.mkdir(parents=True, exist_ok=True)
lbl_dir.mkdir(parents=True, exist_ok=True)

# Đếm ảnh đã có từ lần trước
existing = len(list(img_dir.glob("*.jpg")))
print(f"📦 Đã có sẵn: {existing} negatives từ lần trước")

model = YOLO(MODEL_PATH)
cap   = cv2.VideoCapture(VIDEO_PATH)

if not cap.isOpened():
    print("❌ Không mở được video!")
    exit(1)

total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
fps          = cap.get(cv2.CAP_PROP_FPS)
print(f"📹 Video: {total_frames} frames | {fps:.1f} fps | {total_frames/fps:.1f}s")

frame_idx   = 0
saved_fp    = 0   # FP frames (có prediction)
saved_clean = 0   # Clean frames (không có prediction)
clean_count = 0   # Đếm clean liên tiếp để sample đều

print("🔍 Đang quét...")

while True:
    ret, frame = cap.read()
    if not ret:
        break

    total_saved = saved_fp + saved_clean + existing
    if total_saved >= TARGET:
        print(f"\n🎯 Đạt mục tiêu {TARGET}! Dừng lại.")
        break

    if frame_idx % EVERY_N != 0:
        frame_idx += 1
        continue

    results = model.predict(frame, conf=CONF, verbose=False)
    r = results[0]

    has_pred = (
        (r.boxes is not None and len(r.boxes) > 0) or
        (r.masks is not None and r.masks.data is not None and len(r.masks.data) > 0)
    )

    stem     = f"prenn_v2_f{frame_idx:06d}"
    img_path = img_dir / f"{stem}.jpg"
    lbl_path = lbl_dir / f"{stem}.txt"

    if has_pred:
        # FP frame → lưu tất cả
        cv2.imwrite(str(img_path), frame)
        lbl_path.write_text("", encoding="utf-8")
        saved_fp += 1

    elif SAVE_CLEAN:
        # Clean frame → lưu theo tỉ lệ
        clean_count += 1
        if clean_count % CLEAN_RATIO == 0:
            cv2.imwrite(str(img_path), frame)
            lbl_path.write_text("", encoding="utf-8")
            saved_clean += 1

    total_saved = saved_fp + saved_clean + existing
    if total_saved % 100 == 0 and total_saved > 0:
        print(f"  💾 {total_saved}/{TARGET} | FP: {saved_fp} | Clean: {saved_clean}")

    frame_idx += 1

cap.release()

# Tổng kết
total_imgs = len(list(img_dir.glob("*.jpg")))
print(f"\n{'='*55}")
print(f"✅ HOÀN TẤT!")
print(f"  FP frames mới:     {saved_fp}")
print(f"  Clean frames mới:  {saved_clean}")
print(f"  Tổng negatives:    {total_imgs}")
print(f"  Folder: {OUTPUT_ROOT}")
print(f"{'='*55}")

# ===== TRAIN LAI =====

In [ ]:
# ===== TRAIN LAI =====
print("\\n" + "=" * 55)
print("BAT DAU TRAIN LAI")
print("=" * 55)
from ultralytics import YOLO

from ultralytics import YOLO

model = YOLO("yolov8s-seg.pt")

results = model.train(
    task="segment",
    data=r"D:\Code\Hu_hong_đường\crack-4\data.yaml",
    epochs=200,
    patience=20,
    batch=16,
    imgsz=640,
    device="0",
    workers=4,
    project=r"D:\Code\Hu_hong_đường\crack-4\runs",
    name="crack_seg_small_hardneg",
    exist_ok=True,
    pretrained=True,
    optimizer="auto",
    verbose=True,
    seed=0,
    deterministic=True,
    single_cls=False,
    rect=False,
    cos_lr=False,
    close_mosaic=10,
    amp=True,
    fraction=1.0,
    profile=False,
    freeze=None,
    multi_scale=False,
    overlap_mask=True,
    mask_ratio=4,
    dropout=0.0,
    val=True,
    split="val",
    save=True,
    save_period=-1,
    save_json=False,
    iou=0.7,
    max_det=300,
    half=False,
    plots=True,
    lr0=0.01,
    lrf=0.01,
    momentum=0.937,
    weight_decay=0.0005,
    warmup_epochs=3.0,
    warmup_momentum=0.8,
    warmup_bias_lr=0.0,
    box=7.5,
    cls=0.5,
    dfl=1.5,
    nbs=64,
    hsv_h=0.015,
    hsv_s=0.7,
    hsv_v=0.4,
    degrees=0.0,
    translate=0.1,
    scale=0.5,
    shear=0.0,
    perspective=0.0,
    flipud=0.0,
    fliplr=0.5,
    bgr=0.0,
    mosaic=1.0,
    mixup=0.0,
    cutmix=0.0,
    copy_paste=0.0,
    copy_paste_mode="flip",
    auto_augment="randaugment",
    erasing=0.4,
)
print("\\n" + "=" * 55)
print("TRAIN HOAN TAT!")
print("  Best: D:\\\\Code\\\\Hu_hong_đường\\\\crack-4\\\\runs\\\\crack_seg_hardneg\\\\weights\\\\best.pt")
print("  Test ngay voi conf=0.55 de it FP!")
print("=" * 55)


# test video

In [ ]:
from ultralytics import YOLO
import cv2, os
import numpy as np
from skimage.morphology import skeletonize

# ===== CONFIG =====
MODEL_PATH      = r"D:\Code\Hu_hong_đường\crack-4\runs\crack_seg_small\weights\best.pt"
VIDEO_INPUT_DIR = r"G:\My Drive\Môn Học trên lớp\Kì II năm 3\Học phần  Thị giác máy tính nâng cao\Seg\videos"
OUTPUT_DIR      = r"G:\My Drive\Môn Học trên lớp\Kì II năm 3\Học phần  Thị giác máy tính nâng cao\Seg\videos\output_video"
CONF = 0.25
IOU  = 0.45

# 1 mét = bao nhiêu pixel, chỉnh theo thực tế nếu bạn đã calibrate
PIXEL_PER_METER = 1000.0

os.makedirs(OUTPUT_DIR, exist_ok=True)

model = YOLO(MODEL_PATH)
VIDEO_EXT = (".mp4", ".mov", ".avi", ".mkv")


def measure_crack_length_pixels(mask):
    if mask is None or mask.size == 0 or np.sum(mask) == 0:
        return 0.0

    binary = (mask > 0).astype(np.uint8)
    skel = skeletonize(binary > 0)
    ys, xs = np.where(skel)

    if len(xs) == 0:
        return 0.0

    skel_u8 = skel.astype(np.uint8)
    length = 0.0

    # Đếm cạnh 8-neighborhood một lần
    for y, x in zip(ys, xs):
        if x + 1 < skel_u8.shape[1] and skel_u8[y, x + 1]:
            length += 1.0
        if y + 1 < skel_u8.shape[0] and skel_u8[y + 1, x]:
            length += 1.0
        if x + 1 < skel_u8.shape[1] and y + 1 < skel_u8.shape[0] and skel_u8[y + 1, x + 1]:
            length += np.sqrt(2)
        if x - 1 >= 0 and y + 1 < skel_u8.shape[0] and skel_u8[y + 1, x - 1]:
            length += np.sqrt(2)

    return float(length)


video_files = [
    f for f in os.listdir(VIDEO_INPUT_DIR)
    if os.path.isfile(os.path.join(VIDEO_INPUT_DIR, f))
    and f.lower().endswith(VIDEO_EXT)
]

print(f"🎬 Found {len(video_files)} videos\n")

for file in sorted(video_files):
    in_path  = os.path.join(VIDEO_INPUT_DIR, file)
    out_path = os.path.join(OUTPUT_DIR, f"out_{os.path.splitext(file)[0]}.mp4")

    cap = cv2.VideoCapture(in_path)
    if not cap.isOpened():
        print(f"❌ Cannot open: {file}")
        continue

    fps = cap.get(cv2.CAP_PROP_FPS) or 30
    w   = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    h   = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

    writer = cv2.VideoWriter(
        out_path,
        cv2.VideoWriter_fourcc(*"mp4v"),
        fps,
        (w, h)
    )

    frame_idx = 0
    lengths_m = []

    print(f"▶️ Processing: {file} ({w}x{h} @ {fps:.0f}fps)")

    while True:
        ret, frame = cap.read()
        if not ret:
            break

        results = model.predict(frame, conf=CONF, iou=IOU, verbose=False)
        r = results[0]

        im = r.plot()

        frame_length_px = 0.0

        if r.masks is not None and r.masks.data is not None:
            masks = r.masks.data.cpu().numpy()
            for mask in masks:
                frame_length_px += measure_crack_length_pixels(mask)

        frame_length_m = frame_length_px / PIXEL_PER_METER
        lengths_m.append(frame_length_m)

        avg_length_m = float(np.mean(lengths_m)) if lengths_m else 0.0
        max_length_m = float(np.max(lengths_m)) if lengths_m else 0.0

        overlay = im.copy()
        cv2.rectangle(overlay, (10, 10), (500, 150), (0, 0, 0), -1)
        cv2.addWeighted(overlay, 0.55, im, 0.45, 0, im)

        cv2.putText(im, f"Frame: {frame_idx:05d}",
                    (20, 40), cv2.FONT_HERSHEY_SIMPLEX,
                    0.9, (255, 255, 255), 2, cv2.LINE_AA)

        cv2.putText(im, f"Length: {frame_length_m:.3f} m",
                    (20, 75), cv2.FONT_HERSHEY_SIMPLEX,
                    1.0, (0, 255, 0), 2, cv2.LINE_AA)

        cv2.putText(im, f"Avg: {avg_length_m:.3f} m | Max: {max_length_m:.3f} m",
                    (20, 110), cv2.FONT_HERSHEY_SIMPLEX,
                    0.8, (0, 255, 255), 2, cv2.LINE_AA)

        writer.write(im)
        frame_idx += 1

    cap.release()
    writer.release()

    if lengths_m:
        print(f"✅ Done: {file}")
        print(f"   Frames: {frame_idx}")
        print(f"   Mean crack length: {np.mean(lengths_m):.3f} m")
        print(f"   Max crack length : {np.max(lengths_m):.3f} m")
        print(f"   Output: {out_path}\n")
    else:
        print(f"✅ Done: {file}")
        print(f"   No crack measured")
        print(f"   Output: {out_path}\n")

print("🏁 DONE ALL VIDEOS")

In [ ]:
from ultralytics import YOLO
import cv2, os
import numpy as np
from skimage.morphology import skeletonize
import torch

# ===== CONFIG =====
MODEL_PATH      = r"D:\\Code\\Hu_hong_đường\\crack-4\\runs\\crack_seg_small_hardneg\\weights\\best.pt"
VIDEO_INPUT_DIR = r"G:\\My Drive\\Môn Học trên lớp\\Kì II năm 3\\Học phần  Thị giác máy tính nâng cao\\Seg\\videos"
OUTPUT_DIR      = r"G:\\My Drive\\Môn Học trên lớp\\Kì II năm 3\\Học phần  Thị giác máy tính nâng cao\\Seg\\videos\\output_video"
CONF = 0.45  # Tăng để giảm false positive
IOU  = 0.45

# 1 mét = bao nhiêu pixel, chỉnh theo thực tế nếu bạn đã calibrate
PIXEL_PER_METER = 1000.0

# Filter để giảm nhầm chân/cỏ
MIN_MASK_AREA = 200   # pixel tối thiểu
MIN_ASPECT_RATIO = 3.0  # dài gấp 3 lần rộng

os.makedirs(OUTPUT_DIR, exist_ok=True)

model = YOLO(MODEL_PATH)
VIDEO_EXT = (".mp4", ".mov", ".avi", ".mkv")

def is_valid_crack_mask(mask):
    """Lọc mask thật của vết nứt (bỏ chân/cỏ)"""
    if mask is None or mask.size == 0:
        return False, 0.0
    
    binary = (mask > 0)
    area = np.sum(binary)
    if area < MIN_MASK_AREA:
        return False, 0.0
    
    ys, xs = np.where(binary)
    if len(xs) == 0:
        return False, 0.0
    
    height = ys.max() - ys.min() + 1
    width = xs.max() - xs.min() + 1
    aspect = max(height, width) / min(height, width)
    
    if aspect < MIN_ASPECT_RATIO:
        return False, 0.0
    
    return True, area

def measure_crack_length_pixels(mask):
    """Đo chiều dài pixel từ skeleton"""
    binary = (mask > 0).astype(np.uint8)
    skel = skeletonize(binary)
    ys, xs = np.where(skel)
    
    if len(xs) == 0:
        return 0.0
    
    skel_u8 = skel.astype(np.uint8)
    length = 0.0
    
    for y, x in zip(ys, xs):
        # Đếm unique edges (tránh double count)
        if x + 1 < skel_u8.shape[1] and skel_u8[y, x + 1]:
            length += 1.0
        if y + 1 < skel_u8.shape[0] and skel_u8[y + 1, x]:
            length += 1.0
        if x + 1 < skel_u8.shape[1] and y + 1 < skel_u8.shape[0] and skel_u8[y + 1, x + 1]:
            length += np.sqrt(2)
        if x - 1 >= 0 and y + 1 < skel_u8.shape[0] and skel_u8[y + 1, x - 1]:
            length += np.sqrt(2)
    
    return length

video_files = [
    f for f in os.listdir(VIDEO_INPUT_DIR)
    if os.path.isfile(os.path.join(VIDEO_INPUT_DIR, f))
    and f.lower().endswith(VIDEO_EXT)
]

print(f"🎬 Found {len(video_files)} videos\\n")

for file in sorted(video_files):
    in_path  = os.path.join(VIDEO_INPUT_DIR, file)
    out_path = os.path.join(OUTPUT_DIR, f"out_{os.path.splitext(file)[0]}.mp4")

    cap = cv2.VideoCapture(in_path)
    if not cap.isOpened():
        print(f"❌ Cannot open: {file}")
        continue

    fps = cap.get(cv2.CAP_PROP_FPS) or 30
    w   = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    h   = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

    writer = cv2.VideoWriter(
        out_path,
        cv2.VideoWriter_fourcc(*"mp4v"),
        fps,
        (w, h)
    )

    frame_idx = 0
    lengths_m = []
    valid_masks_count = 0

    print(f"▶️ Processing: {file} ({w}x{h} @ {fps:.0f}fps)")

    while True:
        ret, frame = cap.read()
        if not ret:
            break

        results = model.predict(frame, conf=CONF, iou=IOU, verbose=False)
        r = results[0]

        im = r.plot()  # Vẽ bbox + mask gốc

        frame_length_px = 0.0
        valid_mask_count = 0

        if r.masks is not None and r.masks.data is not None:
            masks = r.masks.data.cpu().numpy()
            for mask in masks:
                is_valid, _ = is_valid_crack_mask(mask)
                if is_valid:
                    valid_mask_count += 1
                    frame_length_px += measure_crack_length_pixels(mask)

        frame_length_m = frame_length_px / PIXEL_PER_METER
        lengths_m.append(frame_length_m)
        if frame_length_m > 0:
            valid_masks_count += 1

        avg_length_m = float(np.mean(lengths_m)) if lengths_m else 0.0
        max_length_m = float(np.max(lengths_m)) if lengths_m else 0.0

        # UI overlay
        overlay = im.copy()
        cv2.rectangle(overlay, (10, 10), (550, 180), (0, 0, 0), -1)
        cv2.addWeighted(overlay, 0.55, im, 0.45, 0, im)

        cv2.putText(im, f"Frame: {frame_idx:05d} | Valid masks: {valid_mask_count}",
                    (20, 40), cv2.FONT_HERSHEY_SIMPLEX,
                    0.75, (255, 255, 255), 2, cv2.LINE_AA)

        cv2.putText(im, f"Length: {frame_length_m:.3f} m (FILTERED)",
                    (20, 75), cv2.FONT_HERSHEY_SIMPLEX,
                    0.9, (0, 255, 0), 2, cv2.LINE_AA)

        cv2.putText(im, f"Avg: {avg_length_m:.3f} m | Max: {max_length_m:.3f} m",
                    (20, 110), cv2.FONT_HERSHEY_SIMPLEX,
                    0.75, (0, 255, 255), 2, cv2.LINE_AA)

        cv2.putText(im, f"CONF: {CONF} | AREA>={MIN_MASK_AREA}px | ASPECT>{MIN_ASPECT_RATIO}",
                    (20, 145), cv2.FONT_HERSHEY_SIMPLEX,
                    0.5, (255, 255, 0), 1, cv2.LINE_AA)

        writer.write(im)
        frame_idx += 1

    cap.release()
    writer.release()

    if lengths_m:
        total_length_m = np.sum(lengths_m)
        print(f"✅ Done: {file}")
        print(f"   Frames processed: {frame_idx}")
        print(f"   Valid crack detections: {valid_masks_count}")
        print(f"   Mean crack length: {np.mean(lengths_m):.3f} m")
        print(f"   Max crack length: {np.max(lengths_m):.3f} m")
        print(f"   Total length: {total_length_m:.3f} m")
        print(f"   Output: {out_path}\\n")
    else:
        print(f"✅ Done: {file}")
        print(f"   No valid cracks detected")
        print(f"   Output: {out_path}\\n")

print("🏁 DONE ALL VIDEOS")


# test ảnh 

In [ ]:
from ultralytics import YOLO
import cv2, os
from collections import defaultdict
import numpy as np
from skimage.morphology import skeletonize
from scipy import ndimage

# ─── CẤU HÌNH ─────────────────────────────────────────────────
MODEL_PATH      = r"D:\Code\Hu_hong_đường\crack-4\runs\crack_seg_small_hardneg\weights\best.pt"
IMAGE_INPUT_DIR = r"G:\My Drive\Môn Học trên lớp\Kì II năm 3\Học phần  Thị giác máy tính nâng cao\Seg\images"
OUTPUT_DIR      = r"G:\My Drive\Môn Học trên lớp\Kì II năm 3\Học phần  Thị giác máy tính nâng cao\Seg\images\output_images"
CONF = 0.25
IOU  = 0.45

# ⭐ Chỉnh size chữ ở đây
FONT_SCALE_TITLE = 0.75   # chữ tiêu đề (tên file + tổng)
FONT_SCALE_CLASS = 0.70   # chữ từng class
FONT_SCALE_LEN   = 0.65   # chữ độ dài crack
FONT_THICK       = 2      # độ đậm nét chữ
BOX_WIDTH        = 420    # chiều rộng nền đen góc trái (tăng cho dòng length)
ROW_HEIGHT       = 34     # khoảng cách giữa các dòng class

# Scale pixel → mét thực tế
PIXEL_PER_METER = 1000    # Điều chỉnh theo camera (1000px = 1m)

os.makedirs(OUTPUT_DIR, exist_ok=True)
model = YOLO(MODEL_PATH)
all_class_names = list(model.names.values())

IMAGE_EXT = (".jpg", ".jpeg", ".png", ".bmp", ".webp",".heic", ".tiff")

def measure_crack_length(masks):
    """Đo tổng chiều dài vết nứt từ tất cả masks trong frame"""
    total_length = 0.0
    for mask in masks:
        if mask is not None:
            binary = (mask > 0).astype(np.uint8)
            skeleton = skeletonize(binary)
            coords = np.column_stack(np.where(skeleton > 0))
            
            if len(coords) > 0:
                visited = np.zeros(skeleton.shape, bool)
                for i in range(len(coords)):
                    y, x = coords[i]
                    if visited[y, x]: continue
                    
                    stack = [(y, x, 0.0)]
                    visited[y, x] = True
                    
                    while stack:
                        cy, cx, dist = stack.pop()
                        for dy in [-1, 0, 1]:
                            for dx in [-1, 0, 1]:
                                if dy == 0 and dx == 0: continue
                                ny, nx = cy + dy, cx + dx
                                if (0 <= ny < skeleton.shape[0] and 
                                    0 <= nx < skeleton.shape[1] and 
                                    skeleton[ny, nx] and not visited[ny, nx]):
                                    step_dist = 1.0 if (dx == 0 or dy == 0) else np.sqrt(2)
                                    visited[ny, nx] = True
                                    stack.append((ny, nx, dist + step_dist))
                    total_length += dist
    return total_length / 2 / PIXEL_PER_METER

# ─── XỬ LÝ ─────────────────────────────────────────────────────
image_files = [f for f in os.listdir(IMAGE_INPUT_DIR)
               if os.path.isfile(os.path.join(IMAGE_INPUT_DIR, f))
               and f.lower().endswith(IMAGE_EXT)]

print(f"📂 Tìm thấy {len(image_files)} ảnh:")
for f in sorted(image_files):
    print(f"   - {f}")
print()

global_counts = defaultdict(int)
global_lengths = []

for file in sorted(image_files):
    in_path  = os.path.join(IMAGE_INPUT_DIR, file)
    out_path = os.path.join(OUTPUT_DIR, f"out_{file}")

    img = cv2.imread(in_path)
    if img is None:
        print(f"❌ Không đọc được: {file}")
        continue

    results = model.predict(img, conf=CONF, iou=IOU, verbose=False)

    # Đếm class + đo độ dài
    img_counts = defaultdict(int)
    img_length = 0.0
    classes_found = set()
    
    for r in results:
        if r.boxes is not None:
            for box in r.boxes:
                name = model.names[int(box.cls)]
                img_counts[name]    += 1
                global_counts[name] += 1
                classes_found.add(name)
        
        # Đo mask nếu có
        if r.masks is not None:
            img_length = measure_crack_length(r.masks.data.cpu().numpy())
    
    global_lengths.append(img_length)
    total_img = sum(img_counts.values()) or 1
    num_classes = len(classes_found)

    # Vẽ bbox + mask lên ảnh
    im = results[0].plot()

    # ── Bảng thống kê mở rộng ──
    total_all = total_img
    box_h = 44 + len(all_class_names) * ROW_HEIGHT + 60  # Tăng cho dòng length
    overlay = im.copy()
    cv2.rectangle(overlay, (5, 5), (BOX_WIDTH, box_h), (0, 0, 0), -1)
    cv2.addWeighted(overlay, 0.50, im, 0.50, 0, im)

    # Dòng tiêu đề
    cv2.putText(im, f"{file} | {num_classes} class | Total: {total_all}",
                (10, 32), cv2.FONT_HERSHEY_SIMPLEX, FONT_SCALE_TITLE,
                (255, 255, 255), FONT_THICK, cv2.LINE_AA)

    # Từng class
    for i, name in enumerate(sorted(all_class_names)):
        cnt = img_counts[name]
        pct = cnt / total_all * 100
        text = f"{name}: {cnt:>4}    {pct:5.1f}%"
        color = (0, 255, 100) if cnt > 0 else (140, 140, 140)
        cv2.putText(im, text, (10, 52 + i * ROW_HEIGHT),
                    cv2.FONT_HERSHEY_SIMPLEX, FONT_SCALE_CLASS,
                    color, FONT_THICK, cv2.LINE_AA)
    
    # Dòng độ dài crack (nổi bật)
    length_text = f"Crack Length: {img_length:.3f}m"
    cv2.putText(im, length_text, (10, box_h-20),
                cv2.FONT_HERSHEY_SIMPLEX, FONT_SCALE_LEN,
                (0, 255, 0), FONT_THICK+1, cv2.LINE_AA)

    cv2.imwrite(out_path, im)

    print(f"✅ {file} [{num_classes} class | {total_img} bbox | Crack: {img_length:.3f}m]")
    for name in sorted(all_class_names):
        if img_counts[name] > 0:
            print(f"   {name}: {img_counts[name]} ({img_counts[name]/total_img*100:.1f}%)")

print()
print("═" * 65)
print("📊 TỔNG KẾT TẤT CẢ ẢNH:")
total_global = sum(global_counts.values()) or 1
avg_length = np.mean(global_lengths)
max_length = np.max(global_lengths)
print(f"  Tổng bbox: {total_global}")
print(f"  TB độ dài crack: {avg_length:.3f}m")
print(f"  Độ dài max: {max_length:.3f}m")
print("  Chi tiết class:")
for name in sorted(all_class_names):
    cnt = global_counts[name]
    print(f"     {name}: {cnt} ({cnt/total_global*100:.1f}%)")
print("═" * 65)
print("🏁 Xong!")